In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import pingouin
import matplotlib.pyplot as plt
import seaborn as sns
import utils, plotting, graph
from dataClass import nwbWrapper

In [2]:
fnames = utils.get_filenames()
data = nwbWrapper(fnames["london"][0], region="OFC", to_load="all")

loading trial data:
loading trials:


100%|██████████| 897/897 [00:01<00:00, 560.85it/s]


loading choice data:


100%|██████████| 2401/2401 [00:01<00:00, 1467.83it/s]


In [14]:
def condition_averaged_spikes(X, y):
    conditions = np.unique(y)
    n_conditions = len(conditions)
    _, n_timesteps, n_neurons = X.shape
    condition_average = np.zeros((n_conditions, n_timesteps, n_neurons))
    for i, cond in enumerate(conditions):
        idx = np.where(y == cond)[0]
        condition_average[i, ...] = np.mean(X[idx, ...], axis=0)
    return condition_average, conditions
        
y_choice = utils.distance_to_value(data.choice_df.graph_distance.values)
X_choice = data.choice_spikes

plans = data.get_plan_spikes(type="psth", min_duration=100, active_prob_threshold=0.2)
y_plan = utils.distance_to_value(plans["df"].graph_distance.values)
X_plan = plans["spikes"]

100%|██████████| 897/897 [00:01<00:00, 789.01it/s]


In [15]:
choice_avg, _ = condition_averaged_spikes(X_choice, y_choice)
plan_avg, _ = condition_averaged_spikes(X_plan, y_plan)

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def scaledPCA(n_components=20):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_components))
    ])
    
def fit_PCA(cond_avg, n_components, bin_size, step_size):
    model = scaledPCA(n_components=n_components)
    n_cond, n_timesteps, n_neurons = cond_avg.shape
    n_bins = int(n_timesteps / step_size)
    X = np.zeros([n_cond, n_bins, n_neurons])
    for i in range(n_bins):
        start = i * step_size
        end = np.minimum(start + bin_size, n_timesteps)
        X[:, i, :] = np.mean(cond_avg[:, start:end, :].astype(float), axis=1)
    X = X.reshape(n_cond * n_bins, n_neurons)
    PCs = model.fit_transform(X)
    PCs = PCs.reshape(n_cond, n_bins, n_components)
    return model, PCs, X, (n_cond, n_bins, n_components)

def projected_response(X, model):
    scaled_X = model.transform(X)

In [21]:
choice_model, choice_pcs, choice_X, _ = fit_PCA(choice_avg, n_components=10, bin_size=150, step_size=10)
plan_model, plan_pcs, plan_X, _ = fit_PCA(plan_avg, n_components=10, bin_size=150, step_size=10)


In [24]:
def projected_response(X, model):
    scaled_X = model["scaler"].transform(X)
    projected_X = model["pca"].transform(scaled_X)
    var_explained = np.diag(np.cov(projected_X.T)) / np.diag(np.cov(scaled_X.T)).sum()    
    return dict(projection=projected_X, var_explained=var_explained)

choice_in_planPC = projected_response(choice_X, plan_model)